In [1]:
!git clone https://github.com/ag-kunal/news-bias-detection.git
%cd news-bias-detection
!pip install -q -r requirements.txt
!mkdir -p fallacy_detection/reference_data
%cd fallacy_detection/reference_data
!git clone -q --depth 1 https://github.com/ChadiHelwe/MAFALDA
!git clone -q --depth 1 https://github.com/pierpaologoffredo/FallacyDetection
!git clone -q --depth 1 https://github.com/causalNLP/logical-fallacy
%cd ../..
!python fallacy_detection/prepare_fallacy_data.py

Cloning into 'news-bias-detection'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 152 (delta 24), reused 138 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 3.27 MiB | 16.98 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/kaggle/working/news-bias-detection
/kaggle/working/news-bias-detection/fallacy_detection/reference_data
/kaggle/working/news-bias-detection
MAFALDA: 837 rows
ElecDeb: 5,091 rows
LOGIC: 3,800 rows

Saved 8,057 rows -> /kaggle/working/news-bias-detection/outputs/fallacy_detection/training/fallacy_sentences.csv
source         is_fallacy
elecdeb        0.0           2273
               1.0           1447
logic_climate  NaN           1075
logic_edu      NaN           2425
mafalda        0.0            529
               1.0            308

Type counts (positives with a harmonised type):
source                     elecdeb  logic_climate  log

In [2]:
%%writefile fallacy_detection/finetune_binary.py

import argparse, numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from sklearn.model_selection import GroupShuffleSplit

def gsplit(d, frac, seed):
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
                .split(d, groups=d[GROUP_COL]))
    return d.iloc[a], d.iloc[b]


TEXT_COL, LABEL_COL, SOURCE_COL, GROUP_COL = "text", "is_fallacy", "source", "group"


def to_binary(x):
    if isinstance(x, str):
        return 0 if x.strip().lower() in {"0", "none", "nothing", "no fallacy", "no_fallacy", "false"} else 1
    return int(x)

ap = argparse.ArgumentParser()
ap.add_argument("--csv", required=True)
ap.add_argument("--model", default="roberta-base")
ap.add_argument("--holdout", default=None, help="source name to hold out as test set")
ap.add_argument("--pooled", action="store_true", help="random 80/20 split instead")
ap.add_argument("--epochs", type=int, default=3)
ap.add_argument("--bs", type=int, default=16)
ap.add_argument("--lr", type=float, default=2e-5)
ap.add_argument("--max_len", type=int, default=128)
ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--save", default=None, help="dir to save model+tokenizer")
args = ap.parse_args()

torch.manual_seed(args.seed); np.random.seed(args.seed)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

df = pd.read_csv(args.csv).dropna(subset=[TEXT_COL, LABEL_COL, GROUP_COL])
df["y"] = df[LABEL_COL].map(to_binary)
print(df.groupby(SOURCE_COL)["y"].agg(["count", "mean"]))

if args.holdout:
    test = df[df[SOURCE_COL] == args.holdout]
    train = df[df[SOURCE_COL] != args.holdout]
    assert len(test) and len(train), "bad --holdout name"
else:
    train, test = gsplit(df, 0.2, args.seed)
train, val = gsplit(train, 0.1, args.seed)
print(f"train {len(train)} | val {len(val)} | test {len(test)}")

tok = AutoTokenizer.from_pretrained(args.model)
model = AutoModelForSequenceClassification.from_pretrained(args.model, num_labels=2).to(dev)

def make_loader(d, shuffle):
    texts, ys = d[TEXT_COL].astype(str).tolist(), d["y"].tolist()
    def collate(idx):
        enc = tok([texts[i] for i in idx], truncation=True, max_length=args.max_len,
                  padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor([ys[i] for i in idx])
        return enc
    return DataLoader(list(range(len(texts))), batch_size=args.bs, shuffle=shuffle, collate_fn=collate)

tr_dl, va_dl, te_dl = make_loader(train, True), make_loader(val, False), make_loader(test, False)

# class-weighted loss for imbalance
pos = train["y"].mean()
w = torch.tensor([1 / (1 - pos), 1 / pos], dtype=torch.float, device=dev); w = w / w.sum() * 2
loss_fn = torch.nn.CrossEntropyLoss(weight=w)

opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.01)
steps = len(tr_dl) * args.epochs
sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
scaler = torch.cuda.amp.GradScaler(enabled=(dev == "cuda"))

@torch.no_grad()
def predict(dl):
    model.eval(); ps, ys = [], []
    for b in dl:
        b = {k: v.to(dev) for k, v in b.items()}
        y = b.pop("labels")
        with torch.autocast(device_type=dev, enabled=(dev == "cuda")):
            logits = model(**b).logits
        ps.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()); ys.append(y.cpu().numpy())
    return np.concatenate(ps), np.concatenate(ys)

def report(p, y, thr=0.5, tag=""):
    pred = (p >= thr).astype(int)
    print(f"{tag} AUC {roc_auc_score(y, p):.3f} | F1 {f1_score(y, pred):.3f} | macroF1 {f1_score(y, pred, average='macro'):.3f} "
          f"| P {precision_score(y, pred, zero_division=0):.3f} | R {recall_score(y, pred):.3f} | flagged {pred.mean():.2%}")

best_auc, best_state = -1, None
for ep in range(args.epochs):
    model.train()
    for b in tr_dl:
        b = {k: v.to(dev) for k, v in b.items()}
        y = b.pop("labels")
        with torch.autocast(device_type=dev, enabled=(dev == "cuda")):
            logits = model(**b).logits
        loss = loss_fn(logits.float(), y)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
    p, y = predict(va_dl); report(p, y, tag=f"epoch {ep+1} val")
    auc = roc_auc_score(y, p)
    if auc > best_auc:
        best_auc = auc; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
p, y = predict(te_dl)
print("\n=== TEST", f"(held-out: {args.holdout})" if args.holdout else "(pooled random split)", "===")
report(p, y, tag="test")
print(f"base rate of fallacy in test: {y.mean():.2%}")

if args.save:
    model.save_pretrained(args.save); tok.save_pretrained(args.save)
    print("saved to", args.save)

Writing fallacy_detection/finetune_binary.py


In [7]:
%cd /kaggle/working/news-bias-detection

[Errno 2] No such file or directory: '/kaggle/working/news-bias-detection'
/news-bias-detection


In [5]:
!python fallacy_detection/finetune_binary.py --csv outputs/fallacy_detection/training/fallacy_sentences.csv --pooled
!python fallacy_detection/finetune_binary.py --csv outputs/fallacy_detection/training/fallacy_sentences.csv --holdout elecdeb
!python fallacy_detection/finetune_binary.py --csv outputs/fallacy_detection/training/fallacy_sentences.csv --holdout mafalda


model.safetensors: downloading bytes: █████▊                |  130MB, 8.90MB/s  epoch 2 val AUC 0.900 | F1 0.766 | macroF1 0.799 | P 0.733 | R 0.803 | flagged 43.63%
epoch 3 val AUC 0.902 | F1 0.777 | macroF1 0.812 | P 0.760 | R 0.796 | flagged 41.73%

=== TEST (pooled random split) ===
test AUC 0.899 | F1 0.759 | macroF1 0.801 | P 0.727 | R 0.794 | flagged 41.11%
base rate of fallacy in test: 37.67%
epoch 1 val AUC 0.812 | F1 0.533 | macroF1 0.650 | P 0.432 | R 0.696 | flagged 41.11%
device: cuda
         count      mean
source                  
elecdeb   3720  0.388978
mafalda    837  0.367981
train 3350 | val 370 | test 837
Loading weights: 100%|██████████████████████| 197/197 [00:00<00:00, 2592.20it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNE

In [7]:
!python fallacy_detection/finetune_binary.py --csv outputs/fallacy_detection/training/fallacy_sentences.csv --pooled --save outputs/fallacy_detection/roberta_binary

device: cuda
         count      mean
source                  
elecdeb   3720  0.388978
mafalda    837  0.367981
train 3288 | val 369 | test 900
Loading weights: 100%|██████████████████████| 197/197 [00:00<00:00, 2706.20it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training

In [9]:
%%writefile score_news_sample.py
import re, argparse, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ap = argparse.ArgumentParser()
ap.add_argument("--model", default="/kaggle/working/news-bias-detection/outputs/fallacy_detection/roberta_binary")
ap.add_argument("--n_articles", type=int, default=300)
ap.add_argument("--n_label", type=int, default=250)
ap.add_argument("--seed", type=int, default=7)
a = ap.parse_args()

df = pd.read_csv("/kaggle/input/datasets/khoshangkashyap/clean-dataset/clean_dataset.csv")
df = df[df.domain == "full"]                     # full articles only
rng = np.random.default_rng(a.seed)
per = a.n_articles // 3
arts = pd.concat([d.sample(min(per, len(d)), random_state=a.seed) for _, d in df.groupby("Bias")])

split = lambda t: [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z\"'])", str(t))]
rows = []
for _, r in arts.iterrows():
    for s in split(r.Text):
        w = len(s.split())
        if 6 <= w <= 60:
            rows.append((r.article_id, r.Bias, s))
S = pd.DataFrame(rows, columns=["article_id", "Bias", "text"])
print("articles:", len(arts), "| sentences:", len(S))

tok = AutoTokenizer.from_pretrained(a.model)
m = AutoModelForSequenceClassification.from_pretrained(a.model).cuda().eval()
ps = []
with torch.no_grad():
    for i in range(0, len(S), 128):
        b = tok(S.text.iloc[i:i+128].tolist(), truncation=True, max_length=128, padding=True, return_tensors="pt").to("cuda")
        with torch.autocast("cuda"):
            ps.append(torch.softmax(m(**b).logits.float(), -1)[:, 1].cpu().numpy())
S["p"] = np.concatenate(ps)

print("\nflagged @0.5:", f"{(S.p>=0.5).mean():.1%}", "| @0.8:", f"{(S.p>=0.8).mean():.1%}")
print("p quantiles:", S.p.quantile([.1,.25,.5,.75,.9]).round(2).to_dict())
print("\nmean p / flagged rate by Bias:")
print(S.groupby("Bias")["p"].agg(mean="mean", flagged=lambda x: (x>=0.5).mean()).round(3).to_string())
art = S.groupby(["article_id", "Bias"])["p"].mean().reset_index()
print("\nper-article mean p by Bias:\n", art.groupby("Bias")["p"].describe().round(3).to_string())

# blind sheet for hand labelling (2 annotators); scores kept separately
samp = S.sample(min(a.n_label, len(S)), random_state=a.seed).reset_index(drop=True)
samp.assign(sid=samp.index)[["sid", "text"]].assign(annotator_A="", annotator_B="").to_csv("/kaggle/working/news-bias-detection/outputs/fallacy_detection/to_label.csv", index=False)
samp.assign(sid=samp.index).to_csv("/kaggle/working/news-bias-detection/outputs/fallacy_detection/to_label_scores_DO_NOT_SHARE.csv", index=False)
print("\nTop 10 flagged sentences:")
for t, p in S.nlargest(10, "p")[["text", "p"]].values: print(f"  {p:.2f}  {t[:160]}")

Overwriting score_news_sample.py


In [10]:
!python score_news_sample.py --n_articles 300 --n_label 250

articles: 300 | sentences: 7354
Loading weights: 100%|██████████████████████| 201/201 [00:00<00:00, 2791.31it/s]

flagged @0.5: 29.6% | @0.8: 16.5%
p quantiles: {0.1: 0.01, 0.25: 0.04, 0.5: 0.18, 0.75: 0.61, 0.9: 0.9}

mean p / flagged rate by Bias:
         mean  flagged
Bias                  
center  0.324    0.295
left    0.314    0.276
right   0.356    0.325

per-article mean p by Bias:
         count   mean    std    min    25%    50%    75%    max
Bias                                                          
center  100.0  0.326  0.140  0.024  0.228  0.310  0.421  0.747
left    100.0  0.328  0.142  0.013  0.248  0.307  0.400  0.983
right    99.0  0.360  0.147  0.040  0.240  0.366  0.438  0.783

Top 10 flagged sentences:
  1.00  Animal Assad would have been history!
  0.99  America’s love affair with confident stupidity continues to reach awful new heights.
  0.99  Millions of people addicted and dying.
  0.99  GEORGE BUSH SPEAKS OUT ON GEORGE FLOYD, RACISM Trump has been a vocal

Now we do extraction and ablation

In [15]:
%%writefile extract_fallacy_features.py
#!/usr/bin/env python

import argparse, os, re, time
import numpy as np, pandas as pd

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="/kaggle/input/datasets/khoshangkashyap/clean-dataset/clean_dataset.csv")
ap.add_argument("--model", default="/kaggle/working/news-bias-detection/outputs/fallacy_detection/roberta_binary")
ap.add_argument("--outdir", default="outputs/fallacy_detection")
ap.add_argument("--limit", type=int, default=0, help="random sample of N articles (0 = all)")
ap.add_argument("--bs", type=int, default=128)
ap.add_argument("--max_len", type=int, default=128)
ap.add_argument("--chunk", type=int, default=20000, help="sentences per checkpoint chunk")
ap.add_argument("--min_words", type=int, default=6)
ap.add_argument("--max_words", type=int, default=60)
ap.add_argument("--dummy", action="store_true", help="random scores, no model (pipeline test only)")
ap.add_argument("--seed", type=int, default=0)
a = ap.parse_args()
os.makedirs(a.outdir, exist_ok=True)

# same splitter + length filter as score_news_sample.py, so features match what was annotated
SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z\"'])")
def split_sentences(t):
    out = []
    for s in SPLIT.split(str(t)):
        s = s.strip()
        if a.min_words <= len(s.split()) <= a.max_words:
            out.append(s)
    return out

df = pd.read_csv(a.data)
if a.limit:
    df = df.sample(min(a.limit, len(df)), random_state=a.seed).reset_index(drop=True)
print(f"articles: {len(df)}")

aid, sidx, sents = [], [], []
for r in df.itertuples():
    for j, s in enumerate(split_sentences(r.Text)):
        aid.append(r.article_id); sidx.append(j); sents.append(s)
S = pd.DataFrame({"article_id": aid, "sent_idx": sidx, "text": sents})
print(f"sentences: {len(S)} | articles with 0 usable sentences: {len(df) - S.article_id.nunique()}")

# ---------- scoring (length-sorted for speed, chunked + resumable) ----------
tag = f"{'dummy' if a.dummy else 'model'}_{len(df)}_{len(S)}"
ck = os.path.join(a.outdir, f"_partial_{tag}.npz")
order = np.argsort(S.text.str.len().values, kind="stable")
p_sorted = np.full(len(S), np.nan, dtype=np.float32)
done = 0
if os.path.exists(ck) and not a.dummy:
    z = np.load(ck); p_sorted, done = z["p"], int(z["done"]); print(f"resuming from {done}/{len(S)}")

if a.dummy:
    p_sorted = np.random.default_rng(a.seed).beta(0.6, 1.2, len(S)).astype(np.float32); done = len(S)
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", dev)
    if dev == "cpu": print("WARNING: CPU scoring of the full dataset will take many hours; use --limit or a GPU")
    tok = AutoTokenizer.from_pretrained(a.model)
    model = AutoModelForSequenceClassification.from_pretrained(a.model).to(dev).eval()
    texts = S.text.values
    t0 = time.time()
    with torch.no_grad():
        for c0 in range(done, len(S), a.chunk):
            c1 = min(c0 + a.chunk, len(S))
            for i in range(c0, c1, a.bs):
                idx = order[i:min(i + a.bs, c1)]
                enc = tok([texts[k] for k in idx], truncation=True, max_length=a.max_len,
                          padding=True, return_tensors="pt").to(dev)
                with torch.autocast(device_type=dev, enabled=(dev == "cuda")):
                    lg = model(**enc).logits
                p_sorted[i:i + len(idx)] = torch.softmax(lg.float(), -1)[:, 1].cpu().numpy()
            np.savez(ck, p=p_sorted, done=c1)
            el = time.time() - t0
            print(f"  {c1}/{len(S)} sentences | {el/60:.1f} min elapsed | ETA {el/(c1-done)*(len(S)-c1)/60:.1f} min", flush=True)

S["p"] = np.nan
S.loc[order, "p"] = p_sorted[:len(S)] if not a.dummy else p_sorted[np.argsort(order)]
assert S.p.notna().all(), "unscored sentences remain"
S[["article_id", "sent_idx", "p"]].to_csv(os.path.join(a.outdir, "sentence_scores.csv.gz"), index=False)

# ---------- article-level aggregation ----------
g = S.groupby("article_id")["p"]
F = pd.DataFrame({
    "fal_n_sent": g.size(),
    "fal_p_mean": g.mean(),
    "fal_p_max": g.max(),
    "fal_p_std": g.std().fillna(0),
    "fal_p_q90": g.quantile(0.9),
})
for t in (0.5, 0.8, 0.9):
    k = int(round(t * 100))
    flag = (S.p >= t).astype(float)
    F[f"fal_share_ge{k}"] = flag.groupby(S.article_id).mean()
F["fal_count_ge90"] = (S.p >= 0.9).groupby(S.article_id).sum()
F["fal_top3_mean"] = S.sort_values("p", ascending=False).groupby("article_id")["p"].apply(lambda x: x.head(3).mean())
F = F.reindex(df.article_id.values)            # articles with no usable sentence -> NaN
F.index.name = "article_id"
no_sent = F.fal_n_sent.isna()
F = F.fillna(0.0).reset_index()
F["fal_no_sentences"] = no_sent.values.astype(int)

keep = [c for c in ["article_id", "Bias", "file", "domain", "group_id", "fold"] if c in df.columns]
out = df[keep].merge(F, on="article_id", how="left")
fn = os.path.join(a.outdir, "article_fallacy_features.csv")
out.to_csv(fn, index=False)

man = pd.DataFrame([{
    "n_articles": len(df), "n_sentences": len(S), "dummy": a.dummy,
    "flagged_ge50": (S.p >= .5).mean(), "flagged_ge80": (S.p >= .8).mean(), "flagged_ge90": (S.p >= .9).mean(),
}])
man.to_csv(os.path.join(a.outdir, "run_manifest.csv"), index=False)
print("\n", man.T.to_string(header=False))
print("\nmean fallacy features by Bias:")
print(out.groupby("Bias")[["fal_p_mean", "fal_p_max", "fal_share_ge90"]].mean().round(3).to_string())
if "domain" in out.columns:
    print("\nby domain:"); print(out.groupby("domain")[["fal_n_sent", "fal_p_mean", "fal_share_ge90"]].mean().round(3).to_string())
print("\nsaved", fn)
if os.path.exists(ck) and not a.dummy and S.p.notna().all():
    os.remove(ck)

Overwriting extract_fallacy_features.py


In [16]:
%%writefile ablation.py
#!/usr/bin/env python
import argparse, os, re, warnings
import numpy as np, pandas as pd
from scipy.stats import kruskal
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")
 
ap = argparse.ArgumentParser()
ap.add_argument("--clean", default="/kaggle/input/datasets/khoshangkashyap/clean-dataset/clean_dataset.csv")
ap.add_argument("--fal", default="outputs/fallacy_detection/article_fallacy_features.csv")
ap.add_argument("--outdir", default="outputs/ablation")
ap.add_argument("--models", default="lr,xgb")
ap.add_argument("--seeds", type=int, default=3, help="repeats for xgboost")
ap.add_argument("--boot", type=int, default=1000)
ap.add_argument("--stage1", default=None)
ap.add_argument("--stage1-keys", default="Title,file")
a = ap.parse_args()
os.makedirs(a.outdir, exist_ok=True)
 
# ------------------------------------------------------------------ data
clean = pd.read_csv(a.clean)
fal = pd.read_csv(a.fal)
FAL = [c for c in fal.columns if c.startswith("fal_")]
df = clean.merge(fal[["article_id"] + FAL], on="article_id", how="inner")
print(f"articles with fallacy features: {len(df)} / {len(clean)}")
df["Text"] = df["Text"].astype(str)
 
# ------------------------------------------------------------------ baseline features (computed from text)
bpath = os.path.join(a.outdir, "baseline_features.csv")
def build_baseline(d):
    t = d["Text"]
    words = t.str.split()
    nw = words.str.len().clip(lower=1)
    nc = t.str.len().clip(lower=1)
    B = pd.DataFrame({"article_id": d.article_id.values})
    B["n_words"] = nw.values
    B["avg_word_len"] = (nc / nw).values
    B["n_sent_rough"] = t.str.count(r"[.!?]+(?:\s|$)").clip(lower=1).values
    B["avg_sent_len"] = (nw / B.n_sent_rough.values).values
    B["ttr"] = words.map(lambda w: len(set(x.lower() for x in w)) / max(len(w), 1)).values
    B["digit_rate"] = (t.str.count(r"\d") / nc).values
    B["upper_word_rate"] = words.map(lambda w: sum(x.isupper() and len(x) > 1 for x in w)).values / nw.values
    for name, pat in {"excl": r"!", "ques": r"\?", "comma": r",", "semi": r";", "colon": r":",
                      "quote": r"[\"“”]", "apos": r"['’]", "dash": r"[—–-]", "paren": r"[()]", "ellipsis": r"\.\.\."}.items():
        B[f"punct_{name}"] = (t.str.count(pat) / nw).values * 100
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    except Exception:
        try:
            from nltk.sentiment.vader import SentimentIntensityAnalyzer
        except Exception:
            SentimentIntensityAnalyzer = None
    if SentimentIntensityAnalyzer is not None:
        sia = SentimentIntensityAnalyzer()
        sc = [sia.polarity_scores(x) for x in t]
        for k in ("compound", "pos", "neg", "neu"):
            B[f"vader_{k}"] = [s[k] for s in sc]
    else:
        print("WARNING: vaderSentiment not installed (pip install vaderSentiment); sentiment features skipped")
    return B
 
if os.path.exists(bpath) and len(pd.read_csv(bpath)) == len(df):
    B = pd.read_csv(bpath); print("loaded cached baseline features")
else:
    print("computing baseline features ..."); B = build_baseline(df); B.to_csv(bpath, index=False)
df = df.merge(B, on="article_id", how="left")
BASE = [c for c in B.columns if c != "article_id"]
 
if a.stage1:
    keys = a.stage1_keys.split(",")
    s1 = pd.read_csv(a.stage1)
    new = [c for c in s1.columns if c not in keys and c not in df.columns]
    n0 = len(df); df = df.merge(s1[keys + new].drop_duplicates(keys), on=keys, how="left")
    BASE += new
    print(f"merged {len(new)} Stage-1 columns; rows with any NaN there: {df[new].isna().any(axis=1).sum()}")
df[BASE + FAL] = df[BASE + FAL].replace([np.inf, -np.inf], np.nan)
df[BASE] = df[BASE].fillna(df[BASE].median())
print(f"baseline features: {len(BASE)} | fallacy features: {len(FAL)}")
 
le = LabelEncoder(); df["y"] = le.fit_transform(df.Bias); classes = list(le.classes_)
rng = np.random.default_rng(0)
df_shuf = df[FAL].sample(frac=1.0, random_state=0).reset_index(drop=True)
for c in FAL: df["shuf_" + c] = df_shuf[c].values
SHUF = ["shuf_" + c for c in FAL]
 
# ------------------------------------------------------------------ models
def make_model(kind, seed):
    if kind == "lr":
        return Pipeline([("sc", StandardScaler()),
                         ("m", LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0))]), False
    try:
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             objective="multi:softprob", tree_method="hist", n_jobs=-1, random_state=seed), True
    except Exception:
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=seed), True
 
def oof_predict(d, cols, kind, seed):
    X = d[cols].values.astype(float); y = d.y.values; pred = np.full(len(d), -1)
    for k in sorted(d.fold.unique()):
        tr, te = d.fold.values != k, d.fold.values == k
        m, use_w = make_model(kind, seed)
        if use_w: m.fit(X[tr], y[tr], sample_weight=compute_sample_weight("balanced", y[tr]))
        else: m.fit(X[tr], y[tr])
        pred[te] = m.predict(X[te])
    return pred
 
def mf1(y, p): return f1_score(y, p, average="macro")
 
# paired bootstrap over story groups
def group_conf(d, pred):
    gi, uniq = pd.factorize(d.group_id)
    C = np.zeros((len(uniq), 3, 3)); np.add.at(C, (gi, d.y.values, pred), 1); return C
def macro_from_C(C):
    tp = np.diagonal(C, axis1=0, axis2=1).astype(float); fp = C.sum(0) - tp; fn = C.sum(1) - tp
    return np.mean(2 * tp / np.maximum(2 * tp + fp + fn, 1e-9))
def boot_diff(d, p_a, p_b, B):
    Ca, Cb = group_conf(d, p_a), group_conf(d, p_b); n = len(Ca); out = np.empty(B)
    for i in range(B):
        idx = rng.integers(0, n, n); out[i] = macro_from_C(Cb[idx].sum(0)) - macro_from_C(Ca[idx].sum(0))
    return out
 
# ------------------------------------------------------------------ univariate check (topic-agnostic)
rows = []
for dom, d in [("all", df)] + list(df.groupby("domain")):
    for c in ["fal_p_mean", "fal_p_max", "fal_share_ge90", "fal_top3_mean"]:
        grp = [g[c].values for _, g in d.groupby("Bias")]
        p = kruskal(*grp).pvalue
        rows.append({"domain": dom, "feature": c, **{f"median_{k}": np.median(g[c]) for k, g in d.groupby("Bias")}, "kruskal_p": p})
U = pd.DataFrame(rows); U.to_csv(os.path.join(a.outdir, "univariate_fallacy_vs_bias.csv"), index=False)
print("\n== fallacy feature vs Bias (Kruskal-Wallis) =="); print(U.round(4).to_string(index=False))
 
# ------------------------------------------------------------------ ablation
sets = {"baseline": BASE, "fallacy_only": FAL, "baseline+fallacy": BASE + FAL, "baseline+shuffled_fallacy": BASE + SHUF}
subsets = {"full_articles": df[df.domain == "full"], "summaries": df[df.domain == "summary"], "all": df}
res, boots = [], []
for sname, d in subsets.items():
    d = d.reset_index(drop=True)
    if len(d) == 0: continue
    maj = mf1(d.y.values, np.full(len(d), d.y.value_counts().idxmax()))
    print(f"\n######## {sname}: n={len(d)} | majority-class macro-F1 = {maj:.3f}")
    for kind in a.models.split(","):
        seeds = range(a.seeds) if kind == "xgb" else [0]
        preds = {}
        for fname, cols in sets.items():
            f1s, ps = [], []
            for sd in seeds:
                p = oof_predict(d, cols, kind, sd); ps.append(p); f1s.append(mf1(d.y.values, p))
            preds[fname] = ps[0]
            fold_f1 = [mf1(d.y.values[d.fold.values == k], ps[0][d.fold.values == k]) for k in sorted(d.fold.unique())]
            per_class = f1_score(d.y.values, ps[0], average=None)
            res.append({"subset": sname, "model": kind, "features": fname, "n": len(d), "macroF1": np.mean(f1s),
                        "macroF1_seed_sd": np.std(f1s), "fold_sd": np.std(fold_f1),
                        **{f"F1_{c}": v for c, v in zip(classes, per_class)}, "majority_macroF1": maj})
            print(f"  {kind:3s} {fname:27s} macroF1 {np.mean(f1s):.4f} (seed sd {np.std(f1s):.4f}, fold sd {np.std(fold_f1):.4f}) "
                  + " ".join(f"{c}:{v:.3f}" for c, v in zip(classes, per_class)), flush=True)
        for comp in ("baseline+fallacy", "baseline+shuffled_fallacy"):
            bd = boot_diff(d, preds["baseline"], preds[comp], a.boot)
            boots.append({"subset": sname, "model": kind, "comparison": f"{comp} - baseline", "delta": bd.mean(),
                          "ci_lo": np.percentile(bd, 2.5), "ci_hi": np.percentile(bd, 97.5),
                          "p_delta<=0": (bd <= 0).mean()})
            print(f"      Δ macro-F1 ({comp} − baseline): {bd.mean():+.4f}  95% CI [{np.percentile(bd,2.5):+.4f}, {np.percentile(bd,97.5):+.4f}]")
 
R = pd.DataFrame(res); Bt = pd.DataFrame(boots)
R.to_csv(os.path.join(a.outdir, "ablation_results.csv"), index=False)
Bt.to_csv(os.path.join(a.outdir, "ablation_bootstrap.csv"), index=False)
print("\n== summary: improvement from adding fallacy features ==")
print(Bt.round(4).to_string(index=False))
print("\nsaved to", a.outdir)
print("Reading guide: a real gain needs a CI above 0 AND to beat the shuffled-fallacy control.")
 

Overwriting ablation.py


In [17]:
!pip install -q vaderSentiment xgboost
!python /kaggle/working/extract_fallacy_features.py --limit 500


articles: 500
sentences: 7889 | articles with 0 usable sentences: 2
device: cuda
Loading weights: 100%|██████████████████████| 201/201 [00:00<00:00, 2696.00it/s]
  7889/7889 sentences | 0.1 min elapsed | ETA 0.0 min

 n_articles         500
n_sentences       7889
dummy            False
flagged_ge50   0.30777
flagged_ge80  0.160857
flagged_ge90  0.104196

mean fallacy features by Bias:
        fal_p_mean  fal_p_max  fal_share_ge90
Bias                                         
center       0.343      0.775           0.105
left         0.387      0.798           0.113
right        0.409      0.825           0.151

by domain:
         fal_n_sent  fal_p_mean  fal_share_ge90
domain                                         
full         24.673       0.332           0.106
summary       2.435       0.472           0.154

saved outputs/fallacy_detection/article_fallacy_features.csv


In [18]:
!python /kaggle/working/extract_fallacy_features.py
!python /kaggle/working/ablation.py

articles: 35010
sentences: 548690 | articles with 0 usable sentences: 375
device: cuda
Loading weights: 100%|██████████████████████| 201/201 [00:00<00:00, 2900.51it/s]
  20000/548690 sentences | 0.1 min elapsed | ETA 2.3 min
  40000/548690 sentences | 0.2 min elapsed | ETA 2.2 min
  60000/548690 sentences | 0.3 min elapsed | ETA 2.2 min
  80000/548690 sentences | 0.4 min elapsed | ETA 2.2 min
  100000/548690 sentences | 0.5 min elapsed | ETA 2.3 min
  120000/548690 sentences | 0.6 min elapsed | ETA 2.3 min
  140000/548690 sentences | 0.8 min elapsed | ETA 2.2 min
  160000/548690 sentences | 0.9 min elapsed | ETA 2.2 min
  180000/548690 sentences | 1.1 min elapsed | ETA 2.2 min
  200000/548690 sentences | 1.2 min elapsed | ETA 2.2 min
  220000/548690 sentences | 1.4 min elapsed | ETA 2.1 min
  240000/548690 sentences | 1.6 min elapsed | ETA 2.1 min
  260000/548690 sentences | 1.9 min elapsed | ETA 2.1 min
  280000/548690 sentences | 2.1 min elapsed | ETA 2.0 min
  300000/548690 sentence